In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from itertools import combinations

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
regular_window = 7
reference_window = 7
case = "Comparison"

## Load data:

In [ ]:
# load retailer data
data = pd.read_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window))

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

In [ ]:
df = data

In [ ]:
# Supongamos que ya tienes el DataFrame cargado como df
# Asegúrate de que 'fecha' esté en formato de fecha
df['fecha'] = pd.to_datetime(df['fecha'])

# Filtramos solo los registros con precios no nulos (si es necesario)
df = df[df['precio'].notna()]

# Obtenemos las tiendas únicas
tiendas = df['tienda'].unique()

# Creamos un diccionario con las fechas únicas por tienda
fechas_por_tienda = {tienda: set(df[df['tienda'] == tienda]['fecha']) for tienda in tiendas}

# Creamos la matriz de coincidencia relativa
coincidencia_relativa = pd.DataFrame(index=tiendas, columns=tiendas, dtype=float)

for tienda_i in tiendas:
    for tienda_j in tiendas:
        interseccion = fechas_por_tienda[tienda_i] & fechas_por_tienda[tienda_j]
        total_i = len(fechas_por_tienda[tienda_i])
        coincidencia_relativa.loc[tienda_i, tienda_j] = len(interseccion) / total_i if total_i > 0 else np.nan

# Creamos la matriz de Jaccard
jaccard_matrix = pd.DataFrame(index=tiendas, columns=tiendas, dtype=float)

for tienda_i in tiendas:
    for tienda_j in tiendas:
        interseccion = fechas_por_tienda[tienda_i] & fechas_por_tienda[tienda_j]
        union = fechas_por_tienda[tienda_i] | fechas_por_tienda[tienda_j]
        jaccard_matrix.loc[tienda_i, tienda_j] = len(interseccion) / len(union) if union else np.nan

# Mostramos los resultados
print("Matriz de coincidencia relativa (fracción de fechas de i que coinciden con j):")
print(coincidencia_relativa.round(3))

print("\nMatriz de sincronización con índice de Jaccard:")
print(jaccard_matrix.round(3))
